# Problem Statement

E-commerce platforms lose billions annually to fraudulent transactions, and traditional fraud detection runs in daily/hourly batches — meaning fraud is often caught after the damage is done. This project builds a real-time anomaly detection system using an unsupervised autoencoder that scores each transaction the instant it occurs, flags suspicious ones, and translates model output into estimated dollars saved — giving business stakeholders an intuitive, live view of fraud prevention instead of a black-box model."

# 1. Importing Dataset

In [127]:
!cat app.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
import time

# Load everything app.py needs — independently of the notebook
autoencoder = tf.keras.models.load_model('autoencoder_model.keras')
X_test = joblib.load('X_test.pkl')
threshold = joblib.load('threshold.pkl')
original_amounts = joblib.load('original_amounts.pkl')

st.title("🍬 Real-Time Fraud Detection Dashboard")
placeholder = st.empty()
total_saved = 0
flagged_count = 0

for i in range(50):
    txn = X_test.iloc[i:i+1]
    recon = autoencoder.predict(txn, verbose=0)
    error = np.mean(np.power(txn.values - recon, 2))
    if error > threshold:
        flagged_count += 1
        total_saved += abs(original_amounts.iloc[i])
    with placeholder.container():
        st.metric("Transactions Checked", i+1)
        st.metric("Flagged as Fraud", flagged_count)
        st.metric("Estimated $ Saved", f"${total_saved:,.2f}")
    time.sleep(0.1)


In [128]:
!ls -la *.pkl *.keras

-rw-r--r-- 1 root root    65255 Jul 30 00:26 autoencoder_model.keras
-rw-r--r-- 1 root root  1836210 Jul 30 00:26 original_amounts.pkl
-rw-r--r-- 1 root root      117 Jul 30 00:26 threshold.pkl
-rw-r--r-- 1 root root 14225283 Jul 30 00:26 X_test.pkl


In [129]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
import time

# Load everything app.py needs — independently of the notebook
autoencoder = tf.keras.models.load_model('autoencoder_model.keras')
X_test = joblib.load('X_test.pkl')
threshold = joblib.load('threshold.pkl')
original_amounts = joblib.load('original_amounts.pkl')

st.title("🍬 Real-Time Fraud Detection Dashboard")
placeholder = st.empty()
total_saved = 0
flagged_count = 0

for i in range(50):
    txn = X_test.iloc[i:i+1]
    recon = autoencoder.predict(txn, verbose=0)
    error = np.mean(np.power(txn.values - recon, 2))
    if error > threshold:
        flagged_count += 1
        total_saved += abs(original_amounts.iloc[i])
    with placeholder.container():
        st.metric("Transactions Checked", i+1)
        st.metric("Flagged as Fraud", flagged_count)
        st.metric("Estimated $ Saved", f"${total_saved:,.2f}")
    time.sleep(0.1)

Overwriting app.py


In [130]:
import zipfile

with zipfile.ZipFile('creditcard.csv.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

In [109]:
!cat app.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
import time

# Load everything app.py needs — independently of the notebook
autoencoder = tf.keras.models.load_model('autoencoder_model.keras')
X_test = joblib.load('X_test.pkl')
threshold = joblib.load('threshold.pkl')
original_amounts = joblib.load('original_amounts.pkl')

st.title("🍬 Real-Time Fraud Detection Dashboard")
placeholder = st.empty()
total_saved = 0
flagged_count = 0

for i in range(50):
    txn = X_test.iloc[i:i+1]
    recon = autoencoder.predict(txn, verbose=0)
    error = np.mean(np.power(txn.values - recon, 2))
    if error > threshold:
        flagged_count += 1
        total_saved += abs(original_amounts.iloc[i])
    with placeholder.container():
        st.metric("Transactions Checked", i+1)
        st.metric("Flagged as Fraud", flagged_count)
        st.metric("Estimated $ Saved", f"${total_saved:,.2f}")
    time.sleep(0.1)


In [110]:
!ls -la *.pkl *.keras

-rw-r--r-- 1 root root    65255 Jul 30 00:19 autoencoder_model.keras
-rw-r--r-- 1 root root  1836210 Jul 30 00:19 original_amounts.pkl
-rw-r--r-- 1 root root      117 Jul 30 00:19 threshold.pkl
-rw-r--r-- 1 root root 14225283 Jul 30 00:19 X_test.pkl


# 2. Explore & Prep Data

In [111]:
import pandas as pd

df = pd.read_csv('creditcard.csv')
print(df.shape)
print(df['Class'].value_counts())

(284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64


In [112]:
df['Amount'] = StandardScaler().fit_transform(df[['Amount']])

In [113]:
from sklearn.preprocessing import StandardScaler

# Keep a copy of the real dollar amounts BEFORE scaling
df['Amount_original'] = df['Amount'].copy()

# Now scale for the model (separate column, nothing overwritten)
df['Amount'] = StandardScaler().fit_transform(df[['Amount']])
df['Time'] = StandardScaler().fit_transform(df[['Time']])

In [114]:
normal = df[df.Class == 0]
fraud = df[df.Class == 1]

from sklearn.model_selection import train_test_split

# Keep original_amounts aligned with X_test's index
X_train, X_test = train_test_split(
    normal.drop(['Class', 'Amount_original'], axis=1), test_size=0.2, random_state=42
)
X_test = pd.concat([X_test, fraud.drop(['Class', 'Amount_original'], axis=1)])

# Pull the matching original amounts using the same index
original_amounts = df.loc[X_test.index, 'Amount_original']

y_test = pd.concat([pd.Series([0]*(len(X_test)-len(fraud))), pd.Series([1]*len(fraud))])

In [115]:
autoencoder.save('autoencoder_model.keras')

import joblib
joblib.dump(X_test, 'X_test.pkl')
joblib.dump(threshold, 'threshold.pkl')
joblib.dump(original_amounts, 'original_amounts.pkl')

['original_amounts.pkl']

# 3. Build the Autoencoder

In [116]:
import tensorflow as tf
from tensorflow.keras import layers, models

input_dim = X_train.shape[1]

autoencoder = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(20, activation='relu'),
    layers.Dense(14, activation='relu'),
    layers.Dense(7, activation='relu'),   # bottleneck: forces compression
    layers.Dense(14, activation='relu'),
    layers.Dense(20, activation='relu'),
    layers.Dense(input_dim, activation='linear')  # reconstruct original shape
])

autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_30 (Dense)                │ (None, 20)             │           620 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 14)             │           294 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 7)              │           105 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ (None, 14)             │           112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 20)             │           300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 30)             │           630 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,061 (8.05 KB)

 Trainable params: 2,061 (8.05 KB)

 Non-trainable params: 0 (0.00 B)

In [117]:
history = autoencoder.fit(
    X_train, X_train,   # input = target! (it's learning to recreate its own input)
    epochs=20,
    batch_size=256,
    validation_split=0.1
)

Epoch 1/20
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.6657 - val_loss: 0.4939
Epoch 2/20
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4270 - val_loss: 0.3903
Epoch 3/20
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3624 - val_loss: 0.3502
Epoch 4/20
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3368 - val_loss: 0.3308
Epoch 5/20
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3237 - val_loss: 0.3217
Epoch 6/20
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.3154 - val_loss: 0.3154
Epoch 7/20
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.3096 - val_loss: 0.3095
Epoch 8/20
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3045 - val_loss: 0.3048
Epoch 9/20
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2988 - val_loss: 0.2982
Epoch 10/20
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2935 - val_loss: 0.2937
Epoch 11/20
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2888 - val_loss: 0.2906
Epoch 12/20
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

# 4. Set the Anomaly Threshold

In [118]:
import numpy as np

reconstructions = autoencoder.predict(X_test)
mse = np.mean(np.power(X_test - reconstructions, 2), axis=1)

threshold = np.percentile(mse, 95)  # top 5% reconstruction error = flagged
print("Threshold:", threshold)

1793/1793 ━━━━━━━━━━━━━━━━━━━━ 1s 775us/step
Threshold: 0.6904012223412539


In [119]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = (mse > threshold).astype(int)
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, mse))

              precision    recall  f1-score   support

           0       1.00      0.96      0.98     56863
           1       0.14      0.84      0.25       492

    accuracy                           0.96     57355
   macro avg       0.57      0.90      0.61     57355
weighted avg       0.99      0.96      0.97     57355

ROC-AUC: 0.9379663987713159


# 5. Simulate Real-Time Streaming

In [120]:
import time

def stream_transactions(X_test, delay=0.05):
    for i in range(len(X_test)):
        yield X_test.iloc[i:i+1]
        time.sleep(delay)  # simulate transactions arriving over time

In [121]:
for txn in stream_transactions(X_test.head(50)):
    recon = autoencoder.predict(txn, verbose=0)
    error = np.mean(np.power(txn.values - recon, 2))
    status = "FLAGGED" if error > threshold else "normal"
    print(f"Transaction score: {error:.4f} -> {status}")

Transaction score: 0.0504 -> normal
Transaction score: 0.0761 -> normal
Transaction score: 0.2598 -> normal
Transaction score: 0.1167 -> normal
Transaction score: 0.2285 -> normal
Transaction score: 0.1740 -> normal
Transaction score: 0.3715 -> normal
Transaction score: 0.0761 -> normal
Transaction score: 0.1445 -> normal
Transaction score: 0.2058 -> normal
Transaction score: 0.3006 -> normal
Transaction score: 0.1013 -> normal
Transaction score: 0.0736 -> normal
Transaction score: 0.2399 -> normal
Transaction score: 0.0902 -> normal
Transaction score: 0.2030 -> normal
Transaction score: 0.2385 -> normal
Transaction score: 0.1203 -> normal
Transaction score: 0.1935 -> normal
Transaction score: 0.0575 -> normal
Transaction score: 0.3271 -> normal
Transaction score: 0.1758 -> normal
Transaction score: 0.1963 -> normal
Transaction score: 0.1175 -> normal
Transaction score: 0.2681 -> normal
Transaction score: 0.2670 -> normal
Transaction score: 0.1343 -> normal
Transaction score: 0.4476 ->

# Business Dashboard (Streamlit)

In [122]:
!pip install streamlit pyngrok -q

In [123]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import time

st.title("🍬 Real-Time Fraud Detection Dashboard")
placeholder = st.empty()
total_saved = 0
flagged_count = 0

# (load model, threshold, X_test, 'Amount' original values beforehand)
for i in range(50):
    txn = X_test.iloc[i:i+1]
    recon = autoencoder.predict(txn, verbose=0)
    error = np.mean(np.power(txn.values - recon, 2))
    if error > threshold:
        flagged_count += 1
        total_saved += abs(original_amounts.iloc[i])  # unscaled $ amount
    with placeholder.container():
        st.metric("Transactions Checked", i+1)
        st.metric("Flagged as Fraud", flagged_count)
        st.metric("Estimated $ Saved", f"${total_saved:,.2f}")
    time.sleep(0.1)

Overwriting app.py


In [124]:
from pyngrok import ngrok

ngrok.set_auth_token("3HCIkCHGUYY4MZfxmPnFAIqOjUT_GJAFnPujFC8LrG68JuaU")

In [131]:
ngrok.kill()
!pkill -f streamlit
!streamlit run app.py &>/dev/null&
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://oink-snort-aversion.ngrok-free.dev" -> "http://localhost:8501"
